## OASIS1 → NIfTI 轉換流程

OASIS1 raw 是 Analyze 7.5 格式 (.hdr/.img)，header 沒有方向資訊，需要手動補。

1. 載入並檢查原始檔案
2. Squeeze 維度 + 處理 endianness
3. **建立 affine**
4. 重排成 RAS+ 標準方向
5. 置中 origin
6. 儲存 .nii 並驗證
7. 包成函式做批次處理

> OASIS1 採集規格：
sagittal MP-RAGE，128 slices × 256 × 256，slice 厚 1.25 mm，in-plane 1.0 mm

### 1. 設定路徑、載入原始檔，檢查 header
先用單一檔案跑一遍看每步發生什麼，最後一個 cell 才包成批次。

OASIS1 原始 header 通常會有：
- shape 多一個 4D singleton: `(256, 256, 128, 1)`
- affine 是 identity 或無意義值（因為 Analyze 7.5 沒這個欄位）
- dtype 可能是 big-endian `>i2`（看採集工作站）

In [19]:
import nibabel as nib
import nibabel.orientations as nio
import numpy as np
from pathlib import Path
SRC_PATH = r"C:\Lib\datasets\raw\OASIS1\disc1\OAS1_0001_MR1\RAW\OAS1_0001_MR1_mpr-1_anon.hdr"
DST_PATH = r"C:\Lib\datasets\check\OAS1_0001_MR1_mpr-1_anon.nii"

img = nib.load(SRC_PATH)

print(f"原始 shape   : {img.shape}")
print(f"原始 dtype   : {img.get_data_dtype()}  (byteorder={img.get_data_dtype().byteorder})")
print(f"voxel sizes  : {img.header.get_zooms()}")
print(f"原始 affine  :")
print(img.affine)

原始 shape   : (256, 256, 128, 1)
原始 dtype   : >i2  (byteorder=>)
voxel sizes  : (np.float32(1.0), np.float32(1.0), np.float32(1.25), np.float32(0.0))
原始 affine  :
[[  -1.       0.       0.     127.5  ]
 [   0.       1.       0.    -127.5  ]
 [   0.       0.       1.25   -79.375]
 [   0.       0.       0.       1.   ]]


### 2. Squeeze 維度 + 轉 native endian

- `np.squeeze`：把 `(256, 256, 128, 1)` 壓成 `(256, 256, 128)`
- `.newbyteorder("=")`：強制轉成系統 native endian
  - `'<'` = little-endian (Intel/AMD/ARM 預設)
  - `'>'` = big-endian (老 Sun/SGI 工作站)
  - `'='` = native (跟著系統走)

In [20]:
data = np.asarray(img.dataobj)
print(f"載入後       : shape={data.shape}, dtype={data.dtype} (byteorder='{data.dtype.byteorder}')")

data = np.squeeze(data)
print(f"squeeze 後   : shape={data.shape}")

data = np.ascontiguousarray(data, dtype=data.dtype.newbyteorder("="))
print(f"endian 處理後: dtype={data.dtype} (byteorder='{data.dtype.byteorder}')")
print(f"資料範圍     : min={data.min()}, max={data.max()}, mean={data.mean():.1f}")

載入後       : shape=(256, 256, 128, 1), dtype=>i2 (byteorder='>')
squeeze 後   : shape=(256, 256, 128)
endian 處理後: dtype=int16 (byteorder='=')
資料範圍     : min=0, max=4095, mean=305.6


### 3. 建立 raw affine ⭐

OASIS1 是矢狀面採集 (sagittal MP-RAGE)，voxel 軸與解剖方向對應：

| voxel 軸 | 長度 | 解剖方向 | spacing |
|---|---|---|---|
| axis 0 (i) | 256 | **+Y** (Posterior → Anterior) | 1.0 mm |
| axis 1 (j) | 256 | **+Z** (Inferior → Superior) | 1.0 mm |
| axis 2 (k) | 128 | **−X** (Right → Left) | 1.25 mm |

注意 axis 2 是**負** X 方向（往左走），所以 R 矩陣不是對角矩陣：

$$R = \begin{bmatrix} 0 & 0 & -1.25 \\ 1 & 0 & 0 \\ 0 & 1 & 0 \end{bmatrix}$$

世界座標 `(X, Y, Z) = R · (i, j, k)`

In [21]:
aff = np.eye(4)
aff[:3, :3] = [[0, 0, -1.25],   # world X = -1.25 * k
               [1, 0,  0   ],   # world Y =  1.0  * i
               [0, 1,  0   ]]   # world Z =  1.0  * j

print("affine:")
print(aff)
print()
print("== 驗證對應關係 ==")
print(f"voxel (0, 0, 0)     → world {(aff @ [0, 0, 0, 1])[:3]}")
print(f"voxel (255, 0, 0)   → world {(aff @ [255, 0, 0, 1])[:3]}   (i↑ → Y↑ 前方)")
print(f"voxel (0, 255, 0)   → world {(aff @ [0, 255, 0, 1])[:3]}   (j↑ → Z↑ 上方)")
print(f"voxel (0, 0, 127)   → world {(aff @ [0, 0, 127, 1])[:3]}   (k↑ → X↓ 左方)")

affine:
[[ 0.    0.   -1.25  0.  ]
 [ 1.    0.    0.    0.  ]
 [ 0.    1.    0.    0.  ]
 [ 0.    0.    0.    1.  ]]

== 驗證對應關係 ==
voxel (0, 0, 0)     → world [0. 0. 0.]
voxel (255, 0, 0)   → world [  0. 255.   0.]   (i↑ → Y↑ 前方)
voxel (0, 255, 0)   → world [  0.   0. 255.]   (j↑ → Z↑ 上方)
voxel (0, 0, 127)   → world [-158.75    0.      0.  ]   (k↑ → X↓ 左方)


### 4. 重排成 RAS+（資料實際 transpose & flip）

設好 affine 後，資料**在記憶體裡的排列順序還沒變**。`nib.as_closest_canonical` 會根據 affine 實際做 transpose + flip，讓 voxel 軸對齊世界軸：

- axis 0 = X (Left → Right)
- axis 1 = Y (Posterior → Anterior)  
- axis 2 = Z (Inferior → Superior)

shape 會從 `(256, 256, 128)` 變成 `(128, 256, 256)`，因為原本 axis 2 (128 slices, L-R) 變成最前面那個軸。

In [29]:
nii_raw = nib.Nifti1Image(data, aff)
print(f"重排前 shape  : {nii_raw.shape}")
print(f"重排前 zooms  : {nii_raw.header.get_zooms()}")
print(f"重排前 axcodes: {nio.aff2axcodes(nii_raw.affine)}")
print()

nii = nib.as_closest_canonical(nii_raw)
print(f"重排後 shape  : {nii.shape}")
print(f"重排後 zooms  : {nii.header.get_zooms()}")
print(f"重排後 axcodes: {nio.aff2axcodes(nii.affine)}")
print()
print(f"重排後 affine :")
print(nii.affine)

重排前 shape  : (256, 256, 128)
重排前 zooms  : (np.float32(1.0), np.float32(1.0), np.float32(1.25))
重排前 axcodes: ('A', 'S', 'L')

重排後 shape  : (128, 256, 256)
重排後 zooms  : (np.float32(1.25), np.float32(1.0), np.float32(1.0))
重排後 axcodes: ('R', 'A', 'S')

重排後 affine :
[[   1.25    0.      0.   -158.75]
 [   0.      1.      0.      0.  ]
 [   0.      0.      1.      0.  ]
 [   0.      0.      0.      1.  ]]


### 5. 置中 origin

`as_closest_canonical` 重排後 origin 不在影像中心。重新計算讓「影像幾何中心」對應世界 `(0, 0, 0)`：

$$\text{origin} = -R \cdot \frac{\text{shape} - 1}{2}$$

In [31]:
new_aff = nii.affine.copy()
center_voxel = (np.array(nii.shape) - 1) / 2.0
new_aff[:3, 3] = -new_aff[:3, :3] @ center_voxel

print(f"影像中心 voxel index : {center_voxel}")
print(f"置中前 origin        : {nii.affine[:3, 3]}")
print(f"置中後 origin        : {new_aff[:3, 3]}")
print(f"Brainder 預期        : (-79.375, -127.5, -127.5)")
print()
print(f"最終 affine:")
print(new_aff)

影像中心 voxel index : [ 63.5 127.5 127.5]
置中前 origin        : [-158.75    0.      0.  ]
置中後 origin        : [ -79.375 -127.5   -127.5  ]
Brainder 預期        : (-79.375, -127.5, -127.5)

最終 affine:
[[   1.25     0.       0.     -79.375]
 [   0.       1.       0.    -127.5  ]
 [   0.       0.       1.    -127.5  ]
 [   0.       0.       0.       1.   ]]


### 6. 封裝 NIfTI 並儲存

幾個重點：
- 用 `np.asarray(nii.dataobj)` 取資料，**不要用 `get_fdata()`**（後者會強制轉 float64，浪費 4 倍記憶體）
- `set_qform/sform` 都設一樣的 affine，避免下游工具讀到不一致時跳警告
- `code=1` 是 "Scanner Anatomical"，告訴工具：「這是真實掃描座標」

In [24]:
out = nib.Nifti1Image(np.asarray(nii.dataobj), new_aff)
out.set_qform(new_aff, code=1)
out.set_sform(new_aff, code=1)
out.header.set_xyzt_units("mm", "sec")
out.header["descrip"] = b"OASIS1 -> NIfTI (RAS+, centered origin)"

Path(DST_PATH).parent.mkdir(parents=True, exist_ok=True)
nib.save(out, DST_PATH)

print(f"輸出檔案    : {DST_PATH}")
print(f"檔案大小    : {Path(DST_PATH).stat().st_size / 1024:.1f} KB")
print(f"輸出 shape  : {out.shape}")
print(f"輸出 dtype  : {out.get_data_dtype()}")

輸出檔案    : C:\Lib\datasets\check\OAS1_0001_MR1_mpr-1_anon.nii
檔案大小    : 16384.3 KB
輸出 shape  : (128, 256, 256)
輸出 dtype  : int16


### 7. 檢查

- shape 是不是 `(128, 256, 256)`
- zooms 是不是 `(1.25, 1.0, 1.0)`
- qform/sform code 是不是 1
- axis codes 是不是 `('R', 'A', 'S')`

vitamin E marker（受試者左前額會貼的小膠囊，T1 上會亮白）應該出現在**左前額**位置。如果跑到右側或後腦，方向就是錯的。

In [26]:
check = nib.load(DST_PATH)

print(f"shape       : {check.shape}")
print(f"zooms       : {check.header.get_zooms()}")
print(f"qform code  : {check.header['qform_code']}")
print(f"sform code  : {check.header['sform_code']}")
print(f"axcodes     : {nio.aff2axcodes(check.affine)} ")
print(f"dtype       : {check.get_data_dtype()}")
print()
print(f"affine:")
print(check.affine)

shape       : (128, 256, 256)
zooms       : (np.float32(1.25), np.float32(1.0), np.float32(1.0))
qform code  : 1
sform code  : 1
axcodes     : ('R', 'A', 'S') 
dtype       : int16

affine:
[[   1.25     0.       0.     -79.375]
 [   0.       1.       0.    -127.5  ]
 [   0.       0.       1.    -127.5  ]
 [   0.       0.       0.       1.   ]]


### 8. 包成函式 + 批次處理

確認單檔流程沒問題後，把所有步驟包成 `convert_oasis1`，跑整個資料夾。

In [32]:
def convert_oasis1(src_path, dst_path):
    img = nib.load(str(src_path))
    
    # 1) squeeze + native endian
    data = np.squeeze(np.asarray(img.dataobj))
    data = np.ascontiguousarray(data, dtype=data.dtype.newbyteorder("="))
    
    # 2) raw affine
    aff = np.eye(4)
    aff[:3, :3] = [[0, 0, -1.25], [1, 0, 0], [0, 1, 0]]
    
    # 3) 重排成 RAS+
    nii = nib.as_closest_canonical(nib.Nifti1Image(data, aff))
    
    # 4) origin 置中
    new_aff = nii.affine.copy()
    new_aff[:3, 3] = -new_aff[:3, :3] @ ((np.array(nii.shape) - 1) / 2.0)
    
    # 5) 封裝、寫 header、存檔
    out = nib.Nifti1Image(np.asarray(nii.dataobj), new_aff)
    out.set_qform(new_aff, code=1)
    out.set_sform(new_aff, code=1)
    out.header.set_xyzt_units("mm", "sec")
    
    Path(dst_path).parent.mkdir(parents=True, exist_ok=True)
    nib.save(out, str(dst_path))

In [33]:
# 批次轉換整個資料夾
INPUT_DIR  = r"C:\Lib\datasets\test_retest\OAS1_raw"
OUTPUT_DIR = r"C:\Lib\datasets\test_retest\OAS1_raw\reset"

src, dst = Path(INPUT_DIR), Path(OUTPUT_DIR)
hdrs = sorted(src.rglob("*.hdr"))
print(f"找到 {len(hdrs)} 個 .hdr 檔\n")

ok, err = 0, 0
for hdr in hdrs:
    target = dst / hdr.relative_to(src).with_suffix(".nii")
    try:
        convert_oasis1(hdr, target)
        ok += 1
        print(f"[ok]  {hdr.name}")
    except Exception as e:
        err += 1
        print(f"[err] {hdr.name}: {type(e).__name__}: {e}")

print(f"\n完成: {ok} 成功, {err} 失敗")

找到 160 個 .hdr 檔

[ok]  OAS1_0061_MR1_mpr-1_anon.hdr
[ok]  OAS1_0061_MR1_mpr-2_anon.hdr
[ok]  OAS1_0061_MR1_mpr-3_anon.hdr
[ok]  OAS1_0061_MR1_mpr-4_anon.hdr
[ok]  OAS1_0061_MR2_mpr-1_anon.hdr
[ok]  OAS1_0061_MR2_mpr-2_anon.hdr
[ok]  OAS1_0061_MR2_mpr-3_anon.hdr
[ok]  OAS1_0061_MR2_mpr-4_anon.hdr
[ok]  OAS1_0080_MR1_mpr-1_anon.hdr
[ok]  OAS1_0080_MR1_mpr-2_anon.hdr
[ok]  OAS1_0080_MR1_mpr-3_anon.hdr
[ok]  OAS1_0080_MR1_mpr-4_anon.hdr
[ok]  OAS1_0080_MR2_mpr-1_anon.hdr
[ok]  OAS1_0080_MR2_mpr-2_anon.hdr
[ok]  OAS1_0080_MR2_mpr-3_anon.hdr
[ok]  OAS1_0080_MR2_mpr-4_anon.hdr
[ok]  OAS1_0092_MR1_mpr-1_anon.hdr
[ok]  OAS1_0092_MR1_mpr-2_anon.hdr
[ok]  OAS1_0092_MR1_mpr-3_anon.hdr
[ok]  OAS1_0092_MR1_mpr-4_anon.hdr
[ok]  OAS1_0092_MR2_mpr-1_anon.hdr
[ok]  OAS1_0092_MR2_mpr-2_anon.hdr
[ok]  OAS1_0092_MR2_mpr-3_anon.hdr
[ok]  OAS1_0092_MR2_mpr-4_anon.hdr
[ok]  OAS1_0101_MR1_mpr-1_anon.hdr
[ok]  OAS1_0101_MR1_mpr-2_anon.hdr
[ok]  OAS1_0101_MR1_mpr-3_anon.hdr
[ok]  OAS1_0101_MR1_mpr-4_anon.hdr
[ok